# Geospatial Data Quality & Governance Framework — QC Workflow

Rule-based quality assessment of OpenStreetMap (OSM) buildings, roads and points of interest
for the Dammam–Khobar urban area, Saudi Arabia.

| | |
|---|---|
| **Input** | [Geofabrik GCC States](https://download.geofabrik.de/asia/gcc-states.html) extract, Shapefile format, 16-09-2026 |
| **Output** | `GDQ_Rerun.gdb` (study area, raw layers, `QC_Results`, `QC_Violations`) and `QC_Results_rerun.csv` |
| **Requires** | ArcGIS Pro 3.x with ArcPy (tested on ArcGIS Pro 3.1) |
| **Author** | Norah Alhumud |

**How to run**
1. Set `SRC` and `PRJ` in the **Configuration** cell. Nothing else needs editing.
2. Run *Cell → Run All*.
3. The **Validation** cell compares all 15 rules with the published results and stops with an error if any differ.

The original OSM files are only read, never modified. Data © OpenStreetMap contributors, ODbL.

## 1. Configuration
The only cell to edit.

In [ ]:
import os
import arcpy
from collections import Counter

# Folder that holds the Geofabrik GCC States shapefiles
SRC = r"C:\path\to\gcc-states-260916-free.shp"

# Any existing folder; GDQ_Rerun.gdb and the CSV are written here
PRJ = r"C:\path\to\output_folder"

EXTRACT_DATE = "16-09-2026"
GDB_NAME = "GDQ_Rerun.gdb"

## 2. Input checks
Stops with a clear message if a folder or a required shapefile is missing.

In [ ]:
REQUIRED = {
    "admin":     "gis_osm_adminareas_a_free_1.shp",
    "buildings": "gis_osm_buildings_a_free_1.shp",
    "roads":     "gis_osm_roads_free_1.shp",
    "pois":      "gis_osm_pois_free_1.shp",
}

if not os.path.isdir(SRC):
    raise FileNotFoundError(f"Source folder not found: {SRC}\n"
                            "Set SRC in the Configuration cell to the Geofabrik shapefile folder.")
if not os.path.isdir(PRJ):
    raise FileNotFoundError(f"Output folder not found: {PRJ}\n"
                            "Set PRJ in the Configuration cell to an existing folder.")

missing = [f for f in REQUIRED.values() if not os.path.isfile(os.path.join(SRC, f))]
if missing:
    raise FileNotFoundError("Missing shapefiles in SRC:\n  " + "\n  ".join(missing))

print("Source folder :", SRC)
print("Output folder :", PRJ)
print("All", len(REQUIRED), "required shapefiles found.")

## 3. Environment

In [ ]:
GDB = os.path.join(PRJ, GDB_NAME)
if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(PRJ, GDB_NAME)

arcpy.env.workspace = GDB
arcpy.env.overwriteOutput = True
UTM = arcpy.SpatialReference(32639)   # WGS 84 / UTM zone 39N
WGS = arcpy.SpatialReference(4326)    # WGS 84
sc = arcpy.env.scratchGDB

S = lambda key: os.path.join(SRC, REQUIRED[key])   # source shapefile
G = lambda name: os.path.join(GDB, name)           # output feature class

print("ArcGIS Pro:", arcpy.GetInstallInfo()["Version"])
print("Output GDB:", GDB)

## 4. Admin boundaries
Dammam (OSM `12399708`, 1,846.8 km²) and Khobar (OSM `12400302`, 197.7 km²) governorates,
verified by area and location.

In [ ]:
ADMIN = G("Admin_Dammam_Khobar")
arcpy.conversion.ExportFeatures(S("admin"), ADMIN, "osm_id IN ('12399708', '12400302')")

bl = arcpy.management.MakeFeatureLayer(S("buildings"), "rr_bl_admin")
arcpy.management.SelectLayerByLocation(bl, "HAVE_THEIR_CENTER_IN", ADMIN)

print("admin polygons    :", arcpy.management.GetCount(ADMIN)[0])
print("buildings in admin:", arcpy.management.GetCount(bl)[0])

## 5. Urban study area
Building points (`INSIDE`) aggregated at 500 m in UTM 39N; polygons of 1 km² or more are kept.

In [ ]:
pts, agg = os.path.join(sc, "rr_pts"), os.path.join(sc, "rr_agg")
with arcpy.EnvManager(outputCoordinateSystem=UTM):
    arcpy.management.FeatureToPoint(bl, pts, "INSIDE")
arcpy.cartography.AggregatePoints(in_features=pts, out_feature_class=agg,
                                  aggregation_distance="500 Meters")
arcpy.management.AddField(agg, "km2", "DOUBLE")
arcpy.management.CalculateGeometryAttributes(agg, [["km2", "AREA_GEODESIC"]],
                                             area_unit="SQUARE_KILOMETERS")
a = [r[0] for r in arcpy.da.SearchCursor(agg, ["km2"])]

SA = G("Study_Area")
with arcpy.EnvManager(outputCoordinateSystem=WGS):
    arcpy.conversion.ExportFeatures(agg, SA, "km2 >= 1")

kept = [x for x in a if x >= 1]
print(f"aggregated : {len(a)} polygons, {sum(a):.1f} km2")
print(f"study area : {len(kept)} polygons, {sum(kept):.1f} km2")
if len(kept) > 20:
    raise RuntimeError("Too many study-area polygons: the admin selection was not applied.")

## 6. Raw layers inside the study area

In [ ]:
B, R, P = G("Buildings_Raw"), G("Roads_Raw"), G("POIs_Raw")
for key, out in [("buildings", B), ("roads", R), ("pois", P)]:
    lyr = arcpy.management.MakeFeatureLayer(S(key), "rr_" + key)
    arcpy.management.SelectLayerByLocation(lyr, "HAVE_THEIR_CENTER_IN", SA)
    arcpy.conversion.ExportFeatures(lyr, out)

COUNTS = {os.path.basename(x): int(arcpy.management.GetCount(x)[0]) for x in (B, R, P)}
for k, v in COUNTS.items():
    print(f"{k:14} {v:>7,}")
if COUNTS["Buildings_Raw"] > 200000:
    raise RuntimeError("Buildings_Raw is too large: the study-area selection was not applied.")

## 7. Results tables and helpers

In [ ]:
qc, qv = G("QC_Results"), G("QC_Violations")
for t in (qc, qv):
    if arcpy.Exists(t):
        arcpy.management.Delete(t)

arcpy.management.CreateTable(GDB, "QC_Results")
for f, t, l in [("layer", "TEXT", 40), ("rule_id", "TEXT", 10), ("dimension", "TEXT", 20),
                ("rule", "TEXT", 120), ("severity", "TEXT", 10),
                ("checked", "LONG", None), ("failed", "LONG", None)]:
    arcpy.management.AddField(qc, f, t, field_length=l)

arcpy.management.CreateTable(GDB, "QC_Violations")
arcpy.management.AddField(qv, "rule_id", "TEXT", field_length=10)
arcpy.management.AddField(qv, "osm_id", "TEXT", field_length=20)

RES = {}

def log(layer, rid, dim, rule, sev, checked, bad):
    """Write one rule result and its violating osm_ids."""
    bad = [str(o) for o in bad]
    RES[rid] = (checked, len(bad))
    with arcpy.da.InsertCursor(qc, ["layer", "rule_id", "dimension", "rule",
                                    "severity", "checked", "failed"]) as c:
        c.insertRow((layer, rid, dim, rule, sev, checked, len(bad)))
    with arcpy.da.InsertCursor(qv, ["rule_id", "osm_id"]) as c:
        for o in bad:
            c.insertRow((rid, o))
    print(f"{rid:4} {sev:8} checked {checked:>7,}  failed {len(bad):>5,}")

# Geofabrik stores empty text as a single space
txt = lambda v: (v or "").strip()
print("ready")

## 8. Buildings rules (B1–B4)

In [ ]:
bname = "Dammam_Buildings_Raw"
rows = [(o, g) for o, g in arcpy.da.SearchCursor(B, ["osm_id", "SHAPE@"])]
n = len(rows)

cnt = Counter(o for o, _ in rows)
log(bname, "B1", "Uniqueness", "osm_id unique", "Error", n,
    [o for o, k in cnt.items() if k > 1])

chk = os.path.join(sc, "rr_chk")
arcpy.management.CheckGeometry(B, chk)
bad_oid = {r[0] for r in arcpy.da.SearchCursor(chk, ["FEATURE_ID"])}
oid2id = {r[0]: r[1] for r in arcpy.da.SearchCursor(B, ["OID@", "osm_id"])}
log(bname, "B2", "Geometry", "valid geometry (CheckGeometry)", "Error", n,
    [oid2id[i] for i in bad_oid if i in oid2id])

area = [(o, g.getArea("GEODESIC", "SQUAREMETERS") if g else 0) for o, g in rows]
log(bname, "B3", "Plausibility", "area >= 10 m2", "Warning", n,
    [o for o, a_ in area if a_ < 10])
log(bname, "B4", "Plausibility", "area <= 50,000 m2", "Review", n,
    [o for o, a_ in area if a_ > 50000])

## 9. Roads rules (R1–R7)
A road counts as **named** when it has a `name` **or** a `ref` (route number).
The same definition is used in R2a, R2b and R8.

In [ ]:
rname = "Dammam_Roads_Raw"
F = ["osm_id", "fclass", "name", "ref", "oneway", "maxspeed", "layer", "bridge", "tunnel"]
rd = [dict(zip(F, r)) for r in arcpy.da.SearchCursor(R, F)]
n = len(rd)
MAJOR = ("motorway", "trunk", "primary", "secondary")
named = lambda r: bool(txt(r["name"]) or txt(r["ref"]))

cnt = Counter(r["osm_id"] for r in rd)
log(rname, "R1", "Uniqueness", "osm_id unique", "Error", n,
    [o for o, k in cnt.items() if k > 1])

maj = [r for r in rd if r["fclass"] in MAJOR]
R2A_BAD = [r["osm_id"] for r in maj if not named(r)]
log(rname, "R2a", "Completeness", "motorway/trunk/primary/secondary have name or ref",
    "Error", len(maj), R2A_BAD)

ter = [r for r in rd if r["fclass"] == "tertiary"]
log(rname, "R2b", "Completeness", "tertiary has name or ref", "Warning", len(ter),
    [r["osm_id"] for r in ter if not named(r)])

log(rname, "R3", "Validity", "oneway in F/T/B", "Error", n,
    [r["osm_id"] for r in rd if txt(r["oneway"]) not in ("F", "T", "B")])

ms = [r for r in rd if (r["maxspeed"] or 0) > 0]
log(rname, "R4", "Validity", "maxspeed 20-140 when set (0 = not set)", "Error", len(ms),
    [r["osm_id"] for r in ms if not 20 <= r["maxspeed"] <= 140])

tu = [r for r in rd if txt(r["tunnel"]) == "T"]
log(rname, "R5", "Consistency", "tunnel=T implies layer<0", "Warning", len(tu),
    [r["osm_id"] for r in tu if not (r["layer"] or 0) < 0])

br = [r for r in rd if txt(r["bridge"]) == "T"]
log(rname, "R6", "Consistency", "bridge=T implies layer>0", "Warning", len(br),
    [r["osm_id"] for r in br if not (r["layer"] or 0) > 0])

ly = [r for r in rd if (r["layer"] or 0) != 0]
log(rname, "R7", "Consistency", "layer>0 implies bridge, layer<0 implies tunnel", "Warning",
    len(ly), [r["osm_id"] for r in ly
              if (r["layer"] > 0 and txt(r["bridge"]) != "T")
              or (r["layer"] < 0 and txt(r["tunnel"]) != "T")])

## 10. R8 — unnamed major roads with no named major neighbour
An R2a violation is **isolated** when it does not touch (`BOUNDARY_TOUCHES`) any named major road.

- **Published method:** neighbour counts as named if it has a `name`.
- **Unified method (used from this version):** neighbour counts as named if it has a `name` **or** a `ref`,
  the same definition as R2a.

Both are printed so the effect of the change is visible.

In [ ]:
ids = [str(o) for o in R2A_BAD]
viol = arcpy.management.MakeFeatureLayer(
    R, "rr_r8_viol", "osm_id IN (" + ",".join(f"'{i}'" for i in ids) + ")")
MAJ_SQL = "fclass IN ('motorway', 'trunk', 'primary', 'secondary')"

def isolated(named_sql):
    nb = arcpy.management.MakeFeatureLayer(R, "rr_r8_nb", f"{MAJ_SQL} AND ({named_sql})")
    arcpy.management.SelectLayerByLocation(viol, "BOUNDARY_TOUCHES", nb)
    touching = {str(r[0]) for r in arcpy.da.SearchCursor(viol, ["osm_id"])}
    arcpy.management.SelectLayerByAttribute(viol, "CLEAR_SELECTION")
    return [i for i in ids if i not in touching]

R8_NAME_ONLY = isolated("name <> ' '")
R8_UNIFIED = isolated("name <> ' ' OR ref <> ' '")

print(f"R2a violations                       : {len(ids)}")
print(f"isolated, published method (name)   : {len(R8_NAME_ONLY)}")
print(f"isolated, unified method (name|ref)  : {len(R8_UNIFIED)}")
print(f"attached, unified method             : {len(ids) - len(R8_UNIFIED)}")

log(rname, "R8", "Completeness",
    "R2a violations with no named (name or ref) major neighbour", "Review",
    len(ids), R8_UNIFIED)

## 11. POIs rules (P1–P2)
Commercial POIs are the 12 business classes listed below.

In [ ]:
pname = "Dammam_POIs_Raw"
pois = [dict(zip(["osm_id", "fclass", "name"], r))
        for r in arcpy.da.SearchCursor(P, ["osm_id", "fclass", "name"])]

cnt = Counter(p["osm_id"] for p in pois)
log(pname, "P1", "Uniqueness", "osm_id unique", "Error", len(pois),
    [o for o, k in cnt.items() if k > 1])

BIZ = ("supermarket", "restaurant", "fast_food", "cafe", "bakery", "pharmacy",
       "bank", "hotel", "mall", "clothes", "hairdresser", "car_dealership")
com = [p for p in pois if p["fclass"] in BIZ]
log(pname, "P2", "Completeness", "commercial POIs have name", "Warning", len(com),
    [p["osm_id"] for p in com if not txt(p["name"])])

## 12. Export results

In [ ]:
OUT_CSV = os.path.join(PRJ, "QC_Results_rerun.csv")
arcpy.conversion.ExportTable(qc, OUT_CSV)
print("saved:", OUT_CSV)

## 13. Validation against the published results
Compares all 15 rules (checked / failed) with the published report.
**Any difference stops execution with an error.** Reference values are never changed to make the check pass.

In [ ]:
PUBLISHED = {
    "B1": (91212, 0),  "B2": (91212, 1),  "B3": (91212, 25), "B4": (91212, 3),
    "R1": (18403, 0),  "R2a": (686, 252), "R2b": (705, 518), "R3": (18403, 0),
    "R4": (278, 0),    "R5": (68, 42),    "R6": (99, 0),     "R7": (137, 12),
    "R8": (252, 103),  "P1": (1203, 0),   "P2": (685, 213),
}

diffs = []
print(f"{'rule':5}{'published':>14}{'this run':>14}")
for rid, ref in PUBLISHED.items():
    got = RES.get(rid)
    ok = got == ref
    if not ok:
        diffs.append((rid, ref, got))
    print(f"{rid:5}{ref[0]:>8}/{ref[1]:<5}{str(got[0]) if got else '-':>8}/"
          f"{str(got[1]) if got else '-':<5} {'OK' if ok else '<< DIFFERENT'}")

VALID = len(PUBLISHED) - len(diffs)
print(f"\n{VALID} of {len(PUBLISHED)} rules match")
if diffs:
    raise AssertionError("Results differ from the published report:\n" + "\n".join(
        f"  {rid}: published {ref[0]}/{ref[1]}, this run {got}" for rid, ref, got in diffs))

## 14. Execution summary

In [ ]:
print("EXECUTION SUMMARY")
print("-" * 52)
print("Data source     : OpenStreetMap via Geofabrik (GCC States), ODbL")
print("Extract date    :", EXTRACT_DATE)
print("ArcGIS Pro      :", arcpy.GetInstallInfo()["Version"], "(requires ArcGIS Pro 3.x + ArcPy)")
print("Study area      :", len(kept), "polygons,", round(sum(kept), 1), "km2")
print("Features checked:")
for k, v in COUNTS.items():
    print(f"   {k:14} {v:>7,}")
print("Quality rules   :", len(RES))
print("Validation      :", VALID, "of", len(PUBLISHED), "rules match the published report")
print("Outputs:")
print("  ", GDB, "-> Study_Area, Buildings_Raw, Roads_Raw, POIs_Raw, QC_Results, QC_Violations")
print("  ", OUT_CSV)